# 第五节 从二分类扩展到十分类

## 实验目标
通过本案例的学习：

1. 掌握将一个二分类模型改成多分类模型的方法；


## 注意事项

1. 本案例推荐使用Pytorch-1.0.0、CPU运行；

2. 如果您是第一次使用 JupyterLab，请查看[《ModelArts JupyterLab使用指导》](https://support.huaweicloud.com/devtool-modelarts/devtool-modelarts_0012.html)了解使用方法；

3. 如果您在使用 JupyterLab 过程中碰到报错，请参考[《ModelArts JupyterLab常见问题解决办法》](https://support.huaweicloud.com/modelarts_faq/modelarts_05_0185.html)尝试解决问题。

## 实验步骤

## 案例内容介绍
本案例我们将使用Pytorch来实现手写数字识别十分类，实现代码将在上一节二分类的基础上进行修改

### 1. 加载数据集
第一步，我们仍然是要加载数据集，但是与前面的案例都不同，这次是要加载完整的、十个类别的数据集  
加载数据集的代码如下所示，每段代码的含义，请仔细阅读代码中的注释

In [1]:
import os
import torch
import numpy as np
import torchvision.datasets.mnist as mnist

datasets_dir = '../datasets'
if not os.path.exists(datasets_dir):
    os.makedirs(datasets_dir)
import moxing as mox
if not os.path.exists(os.path.join(datasets_dir, 'MNIST_data.zip')):
    mox.file.copy('obs://modelarts-labs-bj4/course/hwc_edu/deep_learning/datasets/MNIST_data.zip', 
                  os.path.join(datasets_dir, 'MNIST_data.zip'))
    os.system('cd %s; unzip MNIST_data.zip' % (datasets_dir))
    
# 读取完整训练样本
train_data = mnist.read_image_file(os.path.join(datasets_dir, 'MNIST_data/raw/train-images-idx3-ubyte')).numpy().astype(np.uint8)
train_label = mnist.read_label_file(os.path.join(datasets_dir, 'MNIST_data/raw/train-labels-idx1-ubyte')).numpy().astype(np.uint8)
# 读取完整测试样本
test_data = mnist.read_image_file(os.path.join(datasets_dir, 'MNIST_data/raw/t10k-images-idx3-ubyte')).numpy().astype(np.uint8)
test_label = mnist.read_label_file(os.path.join(datasets_dir, 'MNIST_data/raw/t10k-labels-idx1-ubyte')).numpy().astype(np.uint8)

print('训练集规模：', len(train_data), '，测试集规模：', len(test_data))

INFO:root:Using MoXing-v1.17.3-

INFO:root:Using OBS-Python-SDK-3.20.7


训练集规模： 60000 ，测试集规模： 10000


In [2]:
train_x = train_data.reshape(-1, 28*28)  # 每个样本变成一个行向量，因为行向量便于计算
train_y = train_label.reshape(-1, 1)

test_x = test_data.reshape(-1, 28*28)  # 每个样本变成一个行向量，因为行向量便于计算
test_y = test_label.reshape(-1, 1)

打乱数据集的顺序

In [3]:
train_data = np.hstack((train_x, train_y))  # np.hstack表示将两个数组进行水平拼接
test_data = np.hstack((test_x, test_y))  # np.hstack表示将两个数组进行水平拼接
np.random.seed(0)
np.random.shuffle(train_data)  # 打乱train_data数组的行顺序
np.random.shuffle(test_data)  # 打乱train_data数组的行顺序
train_x = train_data[:, :-1]  # 重新取出train_x和train_y
train_y = train_data[:, -1].reshape(-1, 1)
test_x = test_data[:, :-1]  # 重新取出train_x和train_y
test_y = test_data[:, -1].reshape(-1, 1)

查看10个样本

In [4]:
from PIL import Image
batch_size = 10  # 查看10个样本
print(train_y[:batch_size].tolist())
batch_img = train_x[0].reshape(28, 28)
for i in range(1, batch_size):
    batch_img = np.hstack((batch_img, train_x[i].reshape(28, 28)))  # 将一批图片水平拼接起来，方便下一步进行显示
Image.fromarray(batch_img)

[[3], [6], [6], [6], [0], [3], [6], [2], [5], [6]]


进行数据归一化

In [5]:
train_x = torch.FloatTensor(train_x) / 255.0
train_y = torch.LongTensor(train_y).squeeze()  # squeeze()函数能将train_y改成向量
test_x = torch.FloatTensor(test_x) / 255.0
test_y = torch.LongTensor(test_y).squeeze()  # squeeze()函数能将test_y改成向量

到此，我们就完成了训练数据的准备工作，可以将以上操作封装成load_data函数，以便后面再次用到

In [6]:
%%writefile ../datasets/MNIST_data/load_data_all.py
def load_data_all(datasets_dir):
    import os
    import torch
    import numpy as np
    import torchvision.datasets.mnist as mnist

    datasets_dir = '../datasets'
    if not os.path.exists(datasets_dir):
        os.makedirs(datasets_dir)
    import moxing as mox
    if not os.path.exists(os.path.join(datasets_dir, 'MNIST_data.zip')):
        mox.file.copy('obs://modelarts-labs-bj4/course/hwc_edu/deep_learning/datasets/MNIST_data.zip', 
                      os.path.join(datasets_dir, 'MNIST_data.zip'))
        os.system('cd %s; unzip MNIST_data.zip' % (datasets_dir))
        
    # 读取完整训练样本
    train_data = mnist.read_image_file(os.path.join(datasets_dir, 'MNIST_data/raw/train-images-idx3-ubyte')).numpy().astype(np.uint8)
    train_label = mnist.read_label_file(os.path.join(datasets_dir, 'MNIST_data/raw/train-labels-idx1-ubyte')).numpy().astype(np.uint8)
    # 读取完整测试样本
    test_data = mnist.read_image_file(os.path.join(datasets_dir, 'MNIST_data/raw/t10k-images-idx3-ubyte')).numpy().astype(np.uint8)
    test_label = mnist.read_label_file(os.path.join(datasets_dir, 'MNIST_data/raw/t10k-labels-idx1-ubyte')).numpy().astype(np.uint8)

    print('训练集规模：', len(train_data), '，测试集规模：', len(test_data))

    train_x = train_data.reshape(-1, 28*28)  # 每个样本变成一个行向量，因为行向量便于计算
    train_y = train_label.reshape(-1, 1)

    test_x = test_data.reshape(-1, 28*28)  # 每个样本变成一个行向量，因为行向量便于计算
    test_y = test_label.reshape(-1, 1)

    train_data = np.hstack((train_x, train_y))  # np.hstack表示将两个数组进行水平拼接
    test_data = np.hstack((test_x, test_y))  # np.hstack表示将两个数组进行水平拼接
    np.random.seed(0)
    np.random.shuffle(train_data)  # 打乱train_data数组的行顺序
    np.random.shuffle(test_data)  # 打乱train_data数组的行顺序
    train_x = train_data[:, :-1]  # 重新取出train_x和train_y
    train_y = train_data[:, -1].reshape(-1, 1)
    test_x = test_data[:, :-1]  # 重新取出train_x和train_y
    test_y = test_data[:, -1].reshape(-1, 1)

    train_x = torch.FloatTensor(train_x) / 255.0
    train_y = torch.LongTensor(train_y).squeeze()
    test_x = torch.FloatTensor(test_x) / 255.0
    test_y = torch.LongTensor(test_y).squeeze()

    return train_x, train_y, test_x, test_y

Overwriting ../datasets/MNIST_data/load_data_all.py


### 2. 定义网络结构和评价函数
这部分基本复用了上一节的代码，唯一的区别就是nn.Linear函数的out_features参数改成了10，意思是该网络将进行10分类

In [7]:
from torch import nn

class Network(nn.Module):
    def __init__(self, num_of_weights):
        torch.manual_seed(0)
        super().__init__()
        self.fc = nn.Linear(in_features=num_of_weights, out_features=10, bias=True)  # 定义一个全连接层
        self.nonlinearity = nn.Sigmoid()
    
    def forward(self, x):  # 加权求和单元和非线性函数单元通过定义计算过程来实现
        z = self.fc(x)
        pred_y = self.nonlinearity(z)
        return pred_y
   
    def evaluate(self, pred_y, true_y):
        pred_labels = torch.argmax(pred_y, dim=1)
        acc = (pred_labels == true_y).float().mean()
        return acc

### 3. 交叉熵损失函数
torch.nn.functional模块中定义了多种损失函数，对于多分类任务，我们使用cross_entropy函数即可，代码如下：

In [8]:
import torch.nn.functional as F
loss_fun = F.cross_entropy

### 4. 一行代码实现梯度下降算法
梯度下降算法的实现直接复用上一节代码即可

In [9]:
net = Network(28*28)
optimizer = torch.optim.SGD(net.parameters(), lr=0.01)

### 5. 实现训练函数
代码与上一节一致

In [10]:
def train(net, train_x, train_y, test_x, test_y, max_epochs=100):
    train_losses = []
    test_losses = []
    train_accs = []
    test_accs = []
    for epoch in range(1, max_epochs + 1):
        net.train()  # 切换为训练模式
        pred_y_train = net.forward(train_x)  # 前向传播
        train_loss = loss_fun(pred_y_train, train_y)  # 计算损失

        # 计算梯度，更新权值
        train_loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if (epoch == 1) or (epoch % 200 == 0):
            net.eval()  # 切换为评价模式，评价模式不计算梯度，计算更快
            pred_y_test = net.forward(test_x)
            test_loss = loss_fun(pred_y_test, test_y)
            train_acc = net.evaluate(pred_y_train, train_y)
            test_acc = net.evaluate(pred_y_test, test_y)
            print('epoch %d, train_loss %.4f, test_loss %.4f, train_acc: %.4f, test_acc: %.4f' % (epoch, train_loss.item(), test_loss.item(), train_acc, test_acc))
    return train_losses, test_losses, train_accs, test_accs

### 6. 开始训练
代码与上一节一致  
训练耗时约260秒

In [11]:
import time
start_time = time.time()
max_epochs = 3000
train_losses, test_losses, train_accs, test_accs = train(net, train_x, train_y, test_x, test_y, max_epochs=max_epochs)
print('cost time: %.1f s' % (time.time() - start_time))

epoch 1, train_loss 2.3115, test_loss 2.3099, train_acc: 0.0557, test_acc: 0.0561

epoch 200, train_loss 2.1927, test_loss 2.1890, train_acc: 0.6141, test_acc: 0.6284

epoch 400, train_loss 2.1060, test_loss 2.1008, train_acc: 0.7249, test_acc: 0.7334

epoch 600, train_loss 2.0410, test_loss 2.0348, train_acc: 0.7614, test_acc: 0.7710

epoch 800, train_loss 1.9916, test_loss 1.9848, train_acc: 0.7817, test_acc: 0.7928

epoch 1000, train_loss 1.9534, test_loss 1.9461, train_acc: 0.7936, test_acc: 0.8052

epoch 1200, train_loss 1.9231, test_loss 1.9156, train_acc: 0.8019, test_acc: 0.8114

epoch 1400, train_loss 1.8985, test_loss 1.8908, train_acc: 0.8078, test_acc: 0.8157

epoch 1600, train_loss 1.8782, test_loss 1.8704, train_acc: 0.8125, test_acc: 0.8209

epoch 1800, train_loss 1.8610, test_loss 1.8532, train_acc: 0.8162, test_acc: 0.8235

epoch 2000, train_loss 1.8464, test_loss 1.8385, train_acc: 0.8189, test_acc: 0.8270

epoch 2200, train_loss 1.8337, test_loss 1.8257, train_acc: 0

到目前为止，我们基于手写数字二分类的代码进行少量修改，就快速实现了手写数字识别的十分类，总共就三个修改点：  
（1）只加载数字0和1 改成 加载所有数据；  
（2）nn.Linear函数的out_features参数改成10；  
（3）将损失函数mse_loss改成cross_entropy；

修改的过程是非常简单的，但从上面的结果可以看到，该模型训练3000个epoch，在手写数字识别十分类的任务上仅仅达到了0.8383的准确率，而在上一节二分类任务上，模型仅训练50个epoch就达到了0.9986的准确率，说明在感知机这样简单的模型上，手写数字识别十分类要比二分类要难。

至此，本案例完成。